In [88]:
from datasets import load_dataset
import pandas as pd
def load_and_preprocess_wikipedia_by_language(language):
    ds = load_dataset(
        "wikimedia/wikipedia",
        f"20231101.{language}",
        split="train",
        streaming=True
    )

    ds = ds.take(600)  # take first 600 rows
    return ds




In [109]:
def split_df_into_chunks(df, text_column, max_words=256, language="unknown"):

    rows = []
    MAX_WORDS = 256

    for row_idx, row in df.iterrows():
        words = str(row[text_column]).split()
        # Split into chunks of max 256 words
        chunks = [
            words[i:i + MAX_WORDS]
            for i in range(0, len(words), MAX_WORDS)
        ]

        for chunk_idx, chunk in enumerate(chunks):
            rows.append({
                "original_row": row_idx,
                "chunk_index": chunk_idx,
                "chunk_text": " ".join(chunk),
                "LANGUAGE": language
            })

    chunked_df = pd.DataFrame(rows)
    return chunked_df[chunked_df["chunk_text"].str.len() > 0]

In [110]:
languages = ["es", "fr", "de", "en", "nl", "it"]
dfs = {}
for lang in languages:
    ds_lang = load_and_preprocess_wikipedia_by_language(lang)
    df_lang = ds_lang.to_pandas()
    dfs[lang] = split_df_into_chunks(df_lang[300:], text_column="text", max_words=256, language=lang)[:2000]


In [111]:
dfs["en"].head()

,original_row,chunk_index,chunk_text,LANGUAGE
0,300,0,Acoustics is a branch of physics that deals wi...,en
1,300,1,"""ultrasonic"" and ""infrasonic"", respectively. E...",en
2,300,2,"theaters including discussion of interference,...",en
3,300,3,"acoustics (Principia, 1687). Age of Enlightenm...",en
4,300,4,"effects, including biological and psychologica...",en


In [112]:
language_map = {
    "es": "Spanish",
    "fr": "French",
    "de": "German",
    "en": "English",
    "nl": "Dutch",
    "it": "Italian"
}

for lang_code, df in dfs.items():
    df["LANGUAGE"] = language_map.get(lang_code, "unknown")

In [113]:
dfs["en"].describe()

,original_row,chunk_index
count,2000.000000,2000.000000
mean,367.626000,10.542500
std,43.129192,9.713618
min,300.000000,0.000000
25%,327.000000,3.000000
50%,367.000000,8.000000
75%,407.000000,15.000000
max,441.000000,52.000000


In [114]:
final_df = pd.concat(dfs.values(), ignore_index=True)
final_df["label"] = 0
final_df.head()

,original_row,chunk_index,chunk_text,LANGUAGE,label
0,300,0,Las comelináceas (nombre científico Commelinac...,Spanish,0
1,300,1,(con forma de V en el corte transversal). Much...,Spanish,0
2,300,2,"estigma, capitado, con flecos, o 3-lobado. 3 l...",Spanish,0
3,300,3,los géneros de Commelinaceae pertenecen a dos ...,Spanish,0
4,300,4,Hassk. Buforrestia C.B.Clarke Callisia Loefl. ...,Spanish,0


In [115]:
final_df.describe()

,original_row,chunk_index,label
count,12000.000000,12000.000000,12000.0
mean,394.459583,12.816583,0.0
std,61.649361,14.813352,0.0
min,300.000000,0.000000,0.0
25%,342.000000,3.000000,0.0
50%,390.000000,8.000000,0.0
75%,434.000000,18.000000,0.0
max,555.000000,103.000000,0.0


In [116]:
for df in dfs.values():
    print(len(df))

2000
2000
2000
2000
2000
2000


In [117]:
print(len(final_df))

12000


In [118]:
final_df.to_parquet("wikipedia_12000_chunks-test.parquet")

In [122]:
from datasets import load_dataset
import pandas as pd
import os
from tqdm import tqdm



# Output folder
output_dir = "wiki_chunks-test"
os.makedirs(output_dir, exist_ok=True)

# Write each row to a separate txt file
for i, row in tqdm(final_df.iterrows(), total=len(final_df)):
    text = row["chunk_text"]

    file_path = os.path.join(output_dir, f"chunk_{i:06d}.txt")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

100%|██████████| 12000/12000 [00:00<00:00, 12967.28it/s]
